# Preprocessing

In [1]:
from pathlib import Path
import zipfile

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [2]:
DATA_PATH = Path('../data/diabetes+130-us+hospitals+for+years+1999-2008.zip')
CSV_NAME = 'diabetic_data.csv'
TARGET = 'readmitted'

with zipfile.ZipFile(DATA_PATH) as zf:
    with zf.open(CSV_NAME) as f:
        df = pd.read_csv(f, na_values=['?'], keep_default_na=False, low_memory=False)

df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
df.shape, df[TARGET].value_counts(dropna=False)

((101766, 50),
 readmitted
 NO     54864
 >30    35545
 <30    11357
 Name: count, dtype: int64)

In [4]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

missing_summary[missing_summary['missing_count'] > 0]

,missing_count,missing_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [5]:
NON_READMIT_DISCHARGE_IDS = {11, 13, 14, 19, 20, 21}
COUNT_NUMERIC_COLS = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
]
CATEGORICAL_ID_COLS = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
]

def map_diagnosis(code):
    if pd.isna(code):
        return 'Missing'

    code = str(code)
    if code.startswith('V'):
        return 'Supplementary'
    if code.startswith('E'):
        return 'ExternalInjury'

    try:
        code_num = float(code)
    except ValueError:
        return 'Other'

    if code_num == 250:
        return 'Diabetes'
    if 390 <= code_num < 460 or code_num == 785:
        return 'Circulatory'
    if 460 <= code_num < 520 or code_num == 786:
        return 'Respiratory'
    if 520 <= code_num < 580 or code_num == 787:
        return 'Digestive'
    if 580 <= code_num < 630 or code_num == 788:
        return 'Genitourinary'
    if 140 <= code_num < 240:
        return 'Neoplasms'
    if 710 <= code_num < 740:
        return 'Musculoskeletal'
    if 800 <= code_num < 1000:
        return 'Injury'
    return 'Other'

def clean_split(frame, specialty_keepers, constant_cols):
    out = frame.copy()

    out = out[out['discharge_disposition_id'].isin(NON_READMIT_DISCHARGE_IDS) == False]
    out = out[out['gender'] != 'Unknown/Invalid']

    out = out.drop(columns=['encounter_id', 'patient_nbr', 'weight'], errors='ignore')
    out = out.drop(columns=constant_cols, errors='ignore')

    for col in CATEGORICAL_ID_COLS:
        out[col] = out[col].astype(str)

    for col in ['diag_1', 'diag_2', 'diag_3']:
        out[col] = out[col].map(map_diagnosis)

    out['medical_specialty'] = out['medical_specialty'].fillna('Missing')
    out['medical_specialty'] = out['medical_specialty'].where(
        out['medical_specialty'].isin(specialty_keepers),
        'Other'
    )

    return out.reset_index(drop=True)

In [6]:
df_model = df.copy()
df_model = df_model[df_model['discharge_disposition_id'].isin(NON_READMIT_DISCHARGE_IDS) == False]
df_model = df_model[df_model['gender'] != 'Unknown/Invalid']

df_model.shape, df_model[TARGET].value_counts(dropna=False)

((99340, 50),
 readmitted
 NO     52524
 >30    35502
 <30    11314
 Name: count, dtype: int64)

In [7]:
groups = df_model['patient_nbr']
y_full = df_model[TARGET]

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=178)
train_val_idx, test_idx = next(outer_splitter.split(df_model, y_full, groups))

train_val_raw = df_model.iloc[train_val_idx].reset_index(drop=True)
test_raw = df_model.iloc[test_idx].reset_index(drop=True)

inner_groups = train_val_raw['patient_nbr']
inner_y = train_val_raw[TARGET]
inner_splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=178)
train_idx, val_idx = next(inner_splitter.split(train_val_raw, inner_y, inner_groups))

train_raw = train_val_raw.iloc[train_idx].reset_index(drop=True)
val_raw = train_val_raw.iloc[val_idx].reset_index(drop=True)

train_patients = set(train_raw['patient_nbr'])
val_patients = set(val_raw['patient_nbr'])
test_patients = set(test_raw['patient_nbr'])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_raw), len(val_raw), len(test_raw)],
    'unique_patients': [train_raw['patient_nbr'].nunique(), val_raw['patient_nbr'].nunique(), test_raw['patient_nbr'].nunique()]
})

,split,rows,unique_patients
0,train,58891,41498
1,validation,20269,14247
2,test,20180,14242


In [8]:
train_raw[TARGET].value_counts(normalize=True).rename('train_pct'), \
val_raw[TARGET].value_counts(normalize=True).rename('val_pct'), \
test_raw[TARGET].value_counts(normalize=True).rename('test_pct')

(readmitted
 NO     0.527857
 >30    0.357491
 <30    0.114652
 Name: train_pct, dtype: float64,
 readmitted
 NO     0.528689
 >30    0.358380
 <30    0.112931
 Name: val_pct, dtype: float64,
 readmitted
 NO     0.531318
 >30    0.356046
 <30    0.112636
 Name: test_pct, dtype: float64)

In [9]:
specialty_keepers = set(
    train_raw['medical_specialty']
    .fillna('Missing')
    .value_counts()
    .loc[lambda s: s >= 500]
    .index
)

constant_cols = [
    col for col in train_raw.columns
    if train_raw[col].nunique(dropna=False) == 1
]

specialty_keepers, constant_cols

({'Cardiology',
  'Emergency/Trauma',
  'Family/GeneralPractice',
  'InternalMedicine',
  'Missing',
  'Nephrology',
  'Orthopedics',
  'Orthopedics-Reconstructive',
  'Psychiatry',
  'Radiologist',
  'Surgery-General'},
 ['acetohexamide',
  'examide',
  'citoglipton',
  'glimepiride-pioglitazone',
  'metformin-pioglitazone'])

In [10]:
train_df = clean_split(train_raw, specialty_keepers, constant_cols)
val_df = clean_split(val_raw, specialty_keepers, constant_cols)
test_df = clean_split(test_raw, specialty_keepers, constant_cols)

X_train = train_df.drop(columns=[TARGET])
X_val = val_df.drop(columns=[TARGET])
X_test = test_df.drop(columns=[TARGET])

y_train_raw = train_df[TARGET]
y_val_raw = val_df[TARGET]
y_test_raw = test_df[TARGET]

numeric_cols = [col for col in COUNT_NUMERIC_COLS if col in X_train.columns]
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

X_train.shape, len(numeric_cols), len(categorical_cols)

((58891, 41), 8, 33)

In [11]:
post_clean_missing = pd.DataFrame({
    'missing_count': X_train.isna().sum(),
    'missing_pct': (X_train.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

post_clean_missing[post_clean_missing['missing_count'] > 0].head(10)

,missing_count,missing_pct
payer_code,23393,39.72
race,1294,2.20


In [12]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
X_train_prepared.shape, X_val_prepared.shape, X_test_prepared.shape, label_mapping

((58891, 209),
 (20269, 209),
 (20180, 209),
 {'<30': np.int64(0), '>30': np.int64(1), 'NO': np.int64(2)})